# ACHTUNG!!! GEWÜNSCHTE FEATURES EXPLIZIT IN CONFIG SELECTIEREN!!!

# IMPORT

In [1]:
#!pip install python-dotenv # uncomment in Colab

In [2]:
import wandb # weight and biases logs & plots
wandb.login(key = "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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: btcprojekt2026 (btcprojekt2026-bfh) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
import requests
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

import yfinance as yf
import matplotlib.pyplot as plt
import random
import dotenv


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


# CONFIG

In [4]:
config = {

    # Identity
    "group": "DG_Feat+Momentum+Volatility",  # HIER JEDER EXPERIMENT DEUTLICH BESCHREIBEN (MIT EUREN INITIALEN BITTE)
    "seed": 45,

    # Features
    "features": ["r_lag1", "mu_hat", "sigma_hat", "ma_spread", "rsi", "mom_5", "mom_20", "vol_ratio"], # <----- HIER DIE FEATURES AUSWÄHLEN!!

    # Optimization
    "lr": 5e-5,
    "gamma": 0.99,
    "gae_lambda": 0.95,

    # Architecture
    "activation": "Tanh",
    "hidden_layers": [128,128],

    #Training setup
    "total_steps": 600_000,
    "total_updates": 2_000, # original 2000
    "eval_freq": 5_000, # NOT USED
    "eval_episodes": 10,
    "train_every": 4,
    "train_frac": 0.8,


    # PPO
    "vf_coef": 0.5,
    "ent_coef": 0.0001,
    "clip_eps": 0.2,
    "ppo_epochs": 10,
    "target_kl": 0.01,
    "max_grad_norm": 0.5,

    "num_envs": 8,
    "n_steps": 256,
    "minibatch_size": 64,
    "total_updates": 2000,

    # Logging
    "policy_log_freq": 20,
    "policy_log_samples": 1000,

    # Trading parameters
    "fee": 0.0005,
    "kappa": 0.05, # Test Kappa 0.1
    "slippage_coef": 0.001, # Test 0
    "smoothing_alpha": 0.8, # Test 1
    "max_leverage": 1.0,
    "reward_scale": 1.0,
    "include_turnover": True,
    "initial_equity": 100000.0,

    # Eval
    "eval_episodes": 10,

    # Feature index for policy plots
    "feature_index_prev_return": 0,
    "feature_index_mu_hat": 1
}

config["total_steps"] = (
    config["total_updates"]
    * config["n_steps"]
    * config["num_envs"]
)

# HELPER FUNCTIONS AND CLASSES

## Import BTCUSD from Yahoo Finance

In [5]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timezone

def load_ohlcv(ticker="BTC-USD", start="2017-01-01", end="2026-03-04", interval="1d"): # fixed dataset to ensure comparability - can be extended afterwards

    # Si pas de end → jusqu’à aujourd’hui
    if end is None:
        end = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    # Download
    df = yf.download(
        ticker,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=False,
        progress=False
    )

    # Standardiser colonnes
    df = df.rename(columns={
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume"
    })

    df = df[["close", "high", "low", "open", "volume"]]
    df.index.name = "Date"

    return df.astype(float)


# Usage
df = load_ohlcv()
df.head()

Price,close,high,low,open,volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2017-01-01,998.325012,1003.080017,958.698975,963.658020,147775008.0
2017-01-02,1021.750000,1031.390015,996.702026,998.617004,222184992.0
2017-01-03,1043.839966,1044.079956,1021.599976,1021.599976,185168000.0
2017-01-04,1154.729980,1159.420044,1044.400024,1044.400024,344945984.0
2017-01-05,1013.380005,1191.099976,910.416992,1156.729980,510199008.0


## Feature engineering + toy forecasting features

Complete with previous forecast - Teil 1 Projekt  if needed

Initial features

In [6]:
def add_features_and_forecast(df, ewma_span=20, vol_window=20):
    df = df.copy()
    df["log_close"] = np.log(df["close"])
    df["r"] = df["log_close"].diff()

    # Forecast signal (toy): EWMA mean of returns
    df["mu_hat"] = df["r"].ewm(span=ewma_span, adjust=False).mean().shift(1)

    # Risk estimate: rolling volatility
    df["sigma_hat"] = df["r"].rolling(vol_window).std().shift(1)

    # Lag features
    df["r_lag1"] = df["r"].shift(1)
    df = df.dropna()
    return df

df_feat = add_features_and_forecast(df)
df_feat.head()

Price,close,high,low,open,volume,log_close,r,mu_hat,sigma_hat,r_lag1
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD,,,,,
Date,,,,,,,,,,
2017-01-22,924.672974,937.525024,897.564026,922.205017,116573000.0,6.829440,0.003124,0.005661,0.063561,0.029464
2017-01-23,921.012024,928.265991,916.737976,925.499023,73588600.0,6.825473,-0.003967,0.005419,0.063267,0.003124
2017-01-24,892.687012,924.145020,892.286011,910.677002,111349000.0,6.794236,-0.031237,0.004525,0.062964,-0.003967
2017-01-25,901.541992,903.252014,891.687012,891.924011,120831000.0,6.804107,0.009871,0.001119,0.057847,-0.031237
2017-01-26,917.585999,919.325989,902.223999,902.395020,131958000.0,6.821746,0.017640,0.001953,0.050915,0.009871


Technical indicators

In [7]:
def add_technical_indicators(df):
    df = df.copy()
    close = df["close"].squeeze().astype(float)


    # =====================
    # Moving averages
    # =====================

    df["sma50"] = df["close"].rolling(50).mean()
    df["sma200"] = df["close"].rolling(200).mean()

    # MA spread (stationary trend indicator)
    df["ma_spread"] = df["sma50"] / (df["sma200"] + 1e-8) - 1

    # =====================
    # MACD (normalized)
    # =====================

    # =====================
    # MACD (normalized)
    # =====================


    df["ema12"] = close.ewm(span=12, adjust=False).mean()
    df["ema26"] = close.ewm(span=26, adjust=False).mean()

    df["macd"] = (df["ema12"] - df["ema26"]) / (close + 1e-8)

    # =====================
    # Bollinger bands
    # =====================

    sma20 = df["close"].rolling(20).mean()
    std20 = df["close"].rolling(20).std()

    bb_upper = sma20 + 2 * std20
    bb_lower = sma20 - 2 * std20

    # position inside bands (0 → lower band, 1 → upper band)
    df["bb_pos"] = (df["close"] - bb_lower) / (bb_upper - bb_lower + 1e-8)



    # Momentum
    df["mom_5"] = df["r"].rolling(5).mean()
    df["mom_20"] = df["r"].rolling(20).mean()

    # Volatility regime
    df["vol_ratio"] = df["r"].rolling(10).std() / df["r"].rolling(50).std()

    # =====================
    # RSI
    # =====================

    delta = df["close"].diff()

    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / (avg_loss + 1e-8)

    df["rsi"] = 100 - (100 / (1 + rs))

    # normalize RSI to [0,1]
    df["rsi"] = df["rsi"] / 100.0

    df = df.dropna()

    return df


df_tech = add_technical_indicators(df_feat)

print(df_tech.shape)
df_tech.head()

(3129, 21)


Price,close,high,low,open,volume,log_close,r,mu_hat,sigma_hat,r_lag1,...,sma200,ma_spread,ema12,ema26,macd,bb_pos,mom_5,mom_20,vol_ratio,rsi
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD,,,,,,...,,,,,,,,,,
Date,,,,,,,,,,,,,,,,,,,,,
2017-08-09,3342.469971,3422.760010,3247.669922,3420.399902,1.468960e+09,8.114465,-0.022913,0.018532,0.064513,0.012061,...,1769.261777,0.489922,3074.862334,2862.450696,0.063549,0.915606,0.028684,0.008541,0.882089,0.763906
2017-08-10,3381.280029,3453.449951,3319.469971,3341.840088,1.515110e+09,8.126010,0.011544,0.014585,0.046128,-0.022913,...,1781.544812,0.487420,3122.003518,2900.882498,0.065396,0.895049,0.007741,0.011851,0.870716,0.746896
2017-08-11,3650.620117,3679.719971,3372.120117,3373.820068,2.021190e+09,8.202652,0.076643,0.014295,0.043665,0.011544,...,1795.192853,0.486643,3203.329148,2956.418618,0.067635,1.016508,0.025480,0.013083,0.779270,0.768208
2017-08-12,3884.709961,3949.919922,3613.699951,3650.629883,2.219590e+09,8.264804,0.062151,0.020233,0.045180,0.076643,...,1810.152968,0.486950,3308.156966,3025.180940,0.072844,1.058787,0.027897,0.017630,0.761881,0.836610
2017-08-13,4073.260010,4208.390137,3857.800049,3880.040039,3.159090e+09,8.312199,0.047395,0.024225,0.045320,0.062151,...,1826.011558,0.490077,3425.865126,3102.816427,0.079310,1.051994,0.034964,0.019554,0.769389,0.850338


Fear and greed index

In [8]:
def add_fear_and_greed(df, limit=0):
    '''
    Fear and greed from alternative.me - dates available back to 1.2.2018
    https://alternative.me/crypto/fear-and-greed-index/#api
    '''

    df = df.copy()

    url = f"https://api.alternative.me/fng/?limit={limit}"
    response = requests.get(url)
    data = response.json()

    df_fng = pd.DataFrame(data['data'])

    df_fng['timestamp'] = pd.to_datetime(df_fng['timestamp'], unit='s')
    df_fng['value'] = pd.to_numeric(df_fng['value'])

    df_fng.set_index('timestamp', inplace=True)
    df_fng.sort_index(ascending=True, inplace=True)

    df_fng_red = df_fng.loc[:, ['value']]
    df_fng_red.columns = ['fng']

    # normalize to [0,1]
    df_fng_red["fng"] = df_fng_red["fng"] / 100.0

    df = pd.concat([df, df_fng_red], axis=1)

    df = df.dropna()

    return df


df_fng = add_fear_and_greed(df_tech, limit=0)

print(df_fng.shape)
df_fng.head()

(2949, 22)


/tmp/ipykernel_6031/2344990431.py:15: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df_fng['timestamp'] = pd.to_datetime(df_fng['timestamp'], unit='s')


,"(close, BTC-USD)","(high, BTC-USD)","(low, BTC-USD)","(open, BTC-USD)","(volume, BTC-USD)","(log_close, )","(r, )","(mu_hat, )","(sigma_hat, )","(r_lag1, )",...,"(ma_spread, )","(ema12, )","(ema26, )","(macd, )","(bb_pos, )","(mom_5, )","(mom_20, )","(vol_ratio, )","(rsi, )",fng
2018-02-01,9170.540039,10288.799805,8812.280273,10237.299805,9.959400e+09,9.123751,-0.108458,-0.014797,0.064865,0.011295,...,0.786652,10954.873777,11993.576041,-0.113265,0.023716,-0.044236,-0.021084,0.782004,0.348285,0.30
2018-02-02,8830.750000,9142.280273,7796.490234,9142.280273,1.272690e+10,9.085995,-0.037756,-0.023717,0.066780,-0.108458,...,0.759572,10628.085504,11759.292630,-0.128099,-0.001803,-0.057740,-0.024311,0.784131,0.322047,0.15
2018-02-03,9174.910156,9430.750000,8251.629883,8852.120117,7.263790e+09,9.124228,0.038233,-0.025054,0.065898,-0.037756,...,0.730425,10404.520065,11567.856892,-0.126795,0.096310,-0.041602,-0.020308,0.779337,0.228313,0.40
2018-02-04,8277.009766,9334.870117,8031.220215,9175.700195,7.073550e+09,9.021237,-0.102991,-0.019027,0.067197,0.038233,...,0.695680,10077.210789,11324.090438,-0.150644,-0.046659,-0.039935,-0.025631,0.868259,0.242508,0.24
2018-02-05,6955.270020,8364.839844,6756.680176,8270.540039,9.285290e+09,8.847255,-0.173982,-0.027024,0.069395,-0.102991,...,0.660720,9596.912209,11000.474111,-0.201798,-0.134322,-0.076991,-0.025101,1.026269,0.220221,0.11


In [9]:
df_feat = df_fng
print(df_feat.shape)


(2949, 22)


In [10]:
df_feat.head()

,"(close, BTC-USD)","(high, BTC-USD)","(low, BTC-USD)","(open, BTC-USD)","(volume, BTC-USD)","(log_close, )","(r, )","(mu_hat, )","(sigma_hat, )","(r_lag1, )",...,"(ma_spread, )","(ema12, )","(ema26, )","(macd, )","(bb_pos, )","(mom_5, )","(mom_20, )","(vol_ratio, )","(rsi, )",fng
2018-02-01,9170.540039,10288.799805,8812.280273,10237.299805,9.959400e+09,9.123751,-0.108458,-0.014797,0.064865,0.011295,...,0.786652,10954.873777,11993.576041,-0.113265,0.023716,-0.044236,-0.021084,0.782004,0.348285,0.30
2018-02-02,8830.750000,9142.280273,7796.490234,9142.280273,1.272690e+10,9.085995,-0.037756,-0.023717,0.066780,-0.108458,...,0.759572,10628.085504,11759.292630,-0.128099,-0.001803,-0.057740,-0.024311,0.784131,0.322047,0.15
2018-02-03,9174.910156,9430.750000,8251.629883,8852.120117,7.263790e+09,9.124228,0.038233,-0.025054,0.065898,-0.037756,...,0.730425,10404.520065,11567.856892,-0.126795,0.096310,-0.041602,-0.020308,0.779337,0.228313,0.40
2018-02-04,8277.009766,9334.870117,8031.220215,9175.700195,7.073550e+09,9.021237,-0.102991,-0.019027,0.067197,0.038233,...,0.695680,10077.210789,11324.090438,-0.150644,-0.046659,-0.039935,-0.025631,0.868259,0.242508,0.24
2018-02-05,6955.270020,8364.839844,6756.680176,8270.540039,9.285290e+09,8.847255,-0.173982,-0.027024,0.069395,-0.102991,...,0.660720,9596.912209,11000.474111,-0.201798,-0.134322,-0.076991,-0.025101,1.026269,0.220221,0.11


## Train/Test split (time-based)

In [11]:
df_feat.columns = [c[0] if isinstance(c, tuple) else c for c in df_feat.columns]


In [12]:
# -----------------------------
# Train / Test split
# -----------------------------

features = config["features"]

n = len(df_feat)
split = int(config["train_frac"] * n)

df_train = df_feat.iloc[:split].reset_index(drop=True)
df_test  = df_feat.iloc[split:].reset_index(drop=True)
df_train.columns = [c[0] if isinstance(c, tuple) else c for c in df_train.columns]
df_test.columns  = [c[0] if isinstance(c, tuple) else c for c in df_test.columns]
# -----------------------------
# Normalisation (features de config seulement)
# -----------------------------

mean = df_train[features].mean()
std  = df_train[features].std()

df_train[features] = (df_train[features] - mean) / (std + 1e-8)
df_test[features]  = (df_test[features] - mean) / (std + 1e-8)

print(len(df_train), len(df_test))

2359 590


## Trading Environment (target position action)

In [13]:
class TradingEnv(gym.Env):

    metadata = {"render_modes": []}

    def __init__(
        self,
        df,
        config,
        forecast_probs=None,
    ):

        super().__init__()

        self.df = df.reset_index(drop=True)

        self.features = config["features"]

        self.fee = float(config["fee"])
        self.kappa = float(config["kappa"])
        self.slippage_coef = float(config["slippage_coef"])
        self.smoothing_alpha = float(config["smoothing_alpha"])
        self.max_leverage = float(config["max_leverage"])
        self.reward_scale = float(config["reward_scale"])
        self.include_turnover = bool(config["include_turnover"])
        self.initial_equity = float(config["initial_equity"])

        self.forecast_probs = forecast_probs
        self.include_forecast = forecast_probs is not None

        self.action_space = spaces.Box(
            low=-self.max_leverage,
            high=self.max_leverage,
            shape=(1,),
            dtype=np.float32
        )

        portfolio_dim = 3
        if self.include_turnover:
            portfolio_dim += 1

        lstm_dim = 1 if self.include_forecast else 0

        obs_dim = len(self.features) + portfolio_dim + lstm_dim

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(obs_dim,),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.t = 1
        self.pos = 0.0
        self.target_pos = 0.0
        self.prev_turnover = 0.0

        self.equity = self.initial_equity
        self.peak = self.initial_equity

        return self._get_obs(), {}

    def _get_obs(self):

        x = self.df.loc[self.t, self.features].values.astype(np.float32)

        equity_norm = np.float32(self.equity / self.initial_equity)
        drawdown = np.float32((self.peak - self.equity) / (self.peak + 1e-8))

        portfolio = [self.pos, equity_norm, drawdown]

        if self.include_turnover:
            portfolio.append(self.prev_turnover)

        obs = np.concatenate([x, np.array(portfolio, dtype=np.float32)])

        if self.include_forecast:
            if self.t < len(self.forecast_probs):
                signal = float(self.forecast_probs[self.t] * 2 - 1)
            else:
                signal = 0.0
            obs = np.concatenate([obs, [signal]])

        return obs

    def step(self, action):
        # 1) Raw target action from policy
        raw_target = float(np.clip(action[0], -self.max_leverage, self.max_leverage))
        # 2) Position smoothing / execution lag
        # pos_new = (1-alpha)*pos_old + alpha*target
        new_pos = (
            (1 - self.smoothing_alpha) * self.pos
            + self.smoothing_alpha * raw_target
        )

        new_pos = float(np.clip(new_pos, -self.max_leverage, self.max_leverage))
        # 3) Market data for current step
        r_t = float(self.df.loc[self.t, "r"])
        sigma_t = float(self.df.loc[self.t, "sigma_hat"])

        if not np.isfinite(sigma_t):
            sigma_t = 0.0

        # 4) PnL from PREVIOUS position
        pnl = self.pos * r_t
        # 5) Trading turnover
        turnover = abs(new_pos - self.pos)

        # 6) Transaction cost
        cost = self.fee * turnover
        # 7) Slippage / market impact
        # More turnover + more volatility => more execution cost
        slippage = self.slippage_coef * turnover * (1 + sigma_t)
        # 8) Risk penalty
        # Penalize large positions when volatility is high
        risk_pen = self.kappa * (self.pos**2) * sigma_t

        # 9) Final reward
        true_reward = pnl - cost - slippage
        reward = (true_reward - risk_pen) * self.reward_scale

        # 10) Update internal portfolio state
        self.target_pos = raw_target
        self.prev_turnover = turnover
        self.pos = new_pos

        # Keep equity positive in a simple toy way
        self.equity *= float(np.exp(true_reward))
        self.peak = max(self.peak, self.equity)

        # 11) Advance time
        self.t += 1

        terminated = self.t >= len(self.df) - 1
        truncated = False

        info = {
            "pnl": pnl,
            "cost": cost,
            "slippage": slippage,
            "risk_pen": risk_pen,
            "turnover": turnover,
            "position": self.pos,
            "target_position": self.target_pos,
            "equity": self.equity,
            "drawdown": (self.peak - self.equity) / (self.peak + 1e-8),
            "cumulative_return": (self.equity - self.initial_equity) / self.initial_equity,
        }

        return self._get_obs(), float(reward), terminated, truncated, info

## Vectorized env (train)

In [14]:
def make_env(df, seed, config):

    def thunk():

        env = TradingEnv(df, config)

        env.reset(seed=seed)

        return env

    return thunk

env = gym.vector.SyncVectorEnv(
    [make_env(df_train, config["seed"] + i,config)
     for i in range(config["num_envs"])]
)

obs_dim = env.single_observation_space.shape[0]
act_dim = env.single_action_space.shape[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("obs_dim:", obs_dim, "act_dim:", act_dim, "device:", device)

obs_dim: 12 act_dim: 1 device: cuda


## PPO model (Gaussian policy + tanh squash + corrected logprob)

In [15]:
import torch
import torch.nn as nn
from torch.distributions import Normal


class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, config):
        super().__init__()

        activation = getattr(nn, config["activation"])
        hidden_layers = config["hidden_layers"]

        layers = []
        in_dim = obs_dim

        for h in hidden_layers:
            layers.append(nn.Linear(in_dim, h))
            layers.append(activation())
            in_dim = h

        self.net = nn.Sequential(*layers)

        self.mu = nn.Linear(in_dim, act_dim)

        # identique à ton code original
        self.log_std = nn.Parameter(torch.ones(act_dim) * -1.0)

        self.v = nn.Linear(in_dim, 1)

    def forward(self, obs):
        x = self.net(obs)
        mu = self.mu(x)
        std = torch.exp(self.log_std)
        dist = Normal(mu, std)
        value = self.v(x).squeeze(-1)
        return dist, value


# =========================
# SAME FUNCTIONS AS IN INITIAL CODE
# =========================

def squash(u):
    return torch.tanh(u)
# main formula:
# a = f(u)
# log p(a) = log p(u) - log |det(Jacobian)|
# log p(a) ist die gesuchte policy log pi(a|a)

# we have f = tanh
# a = tanh(u)
# da/du = 1 - tanh(u)^2
# da/du = 1 - a²
# we need: log |det(Jacobian)|
# we get: log |det(Jacobian)| = log(1 - tanh(u)^2)
# in code: log_det = torch.log(1.0 - torch.tanh(u).pow(2) + eps).sum(-1)


def logprob_squashed(dist, u):
      # log p(u)
    logp_u = dist.log_prob(u).sum(-1)
        # change-of-variables for tanh
    eps = 1e-6
    log_det = torch.log(1.0 - torch.tanh(u).pow(2) + eps).sum(-1)
    return logp_u - log_det

## GAE

In [16]:
def compute_gae(rewards, dones, values, last_value, gamma, lam):
    """
    rewards:     [T, N]
    dones:       [T, N] (1.0 = true terminal for bootstrap masking)
    values:      [T, N]
    last_value:  [N]
    gamma:       float
    lam:         float
    """

    T, N = rewards.shape

    # sécurité shape (évite broadcasting silencieux)
    if last_value.dim() == 2:
        last_value = last_value.squeeze(-1)

    adv = torch.zeros(T, N, device=values.device)
    gae = torch.zeros(N, device=values.device)

    for t in reversed(range(T)):

        not_done = 1.0 - dones[t]

        if t == T - 1:
            next_value = last_value
        else:
            next_value = values[t + 1]

        delta = rewards[t] + gamma * next_value * not_done - values[t]

        gae = delta + gamma * lam * not_done * gae

        adv[t] = gae

    returns = adv + values

    return returns, adv

## Function Evaluation

In [17]:
def eval_policy(model, df_eval, config, device):

    env_eval = TradingEnv(df_eval, config)

    returns = []

    for _ in range(config["eval_episodes"]):

        obs, _ = env_eval.reset()
        done = False
        ep_ret = 0.0

        while not done:

            obs_t = torch.as_tensor(
                obs, dtype=torch.float32, device=device
            ).unsqueeze(0)

            with torch.no_grad():
                dist, _ = model(obs_t)

                # Deterministic action = mean (mu)
                u = dist.mean # with continuous action, greedy = a=μ(s) (equivalent argmax for DQN)
                a = squash(u).cpu().numpy()[0]

            obs, reward, terminated, truncated, _ = env_eval.step(a)
            done = terminated or truncated

            ep_ret += reward

        returns.append(ep_ret)

    return float(np.mean(returns))

In [18]:
def run_equity_curve(model, df_eval, config, device):

    env_eval = TradingEnv(df_eval, config)

    obs, _ = env_eval.reset()
    done = False

    equity = [env_eval.equity]
    pos_hist = [env_eval.pos]

    while not done:

        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

        with torch.no_grad():
            dist, _ = model(obs_t)
            u = dist.mean
            a = squash(u).cpu().numpy()[0]

        obs, reward, terminated, truncated, _ = env_eval.step(a)

        done = terminated or truncated

        equity.append(env_eval.equity)
        pos_hist.append(env_eval.pos)

    return np.array(equity), np.array(pos_hist)

## Function train_agent

In [19]:
def train_agent(df_train, df_test, config):

    SEED = config["seed"]

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    num_envs = config["num_envs"]
    n_steps = config["n_steps"]
    total_updates = config["total_updates"]

    gamma = config["gamma"]
    lam = config["gae_lambda"]

    ppo_epochs = config["ppo_epochs"]
    minibatch_size = config["minibatch_size"]
    clip_eps = config["clip_eps"]

    vf_coef = config["vf_coef"]
    ent_coef = config["ent_coef"]

    target_kl = config["target_kl"]
    max_grad_norm = config["max_grad_norm"]

    env = gym.vector.SyncVectorEnv(
        [make_env(df_train, SEED + i, config) for i in range(num_envs)]
    )

    obs_dim = env.single_observation_space.shape[0]
    act_dim = env.single_action_space.shape[0]

    model = ActorCritic(obs_dim, act_dim, config).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])

    obs, _ = env.reset(seed=SEED)
    obs = torch.as_tensor(obs, dtype=torch.float32, device=device)

    ep_returns = np.zeros(num_envs)
    ep_turnover = np.zeros(num_envs)

    ep_history = []
    turnover_history = []

    for update in range(total_updates):

        # -------- rollout buffers (prof) --------

        obs_buf = torch.zeros(n_steps, num_envs, obs_dim, device=device)
        u_buf = torch.zeros(n_steps, num_envs, act_dim, device=device)
        logp_buf = torch.zeros(n_steps, num_envs, device=device)
        rew_buf = torch.zeros(n_steps, num_envs, device=device)
        done_buf = torch.zeros(n_steps, num_envs, device=device)
        val_buf = torch.zeros(n_steps, num_envs, device=device)

        # -------- diagnostics (ton code) --------

        pnl_roll = []
        cost_roll = []
        slip_roll = []
        risk_roll = []

        equity_roll = []
        drawdown_roll = []
        position_roll = []

        for t in range(n_steps):

            obs_buf[t] = obs

            with torch.no_grad():

                dist, value = model(obs)

                u = dist.sample()
                a = squash(u)

                logp = logprob_squashed(dist, u)

            u_buf[t] = u
            logp_buf[t] = logp
            val_buf[t] = value.squeeze(-1)

            next_obs, reward, terminated, truncated, infos = env.step(a.cpu().numpy())

            done_env = np.logical_or(terminated, truncated)

            rew_buf[t] = torch.as_tensor(reward, dtype=torch.float32, device=device)
            done_buf[t] = torch.as_tensor(done_env, dtype=torch.float32, device=device)

            # -------- logging diagnostics --------

            if isinstance(infos, dict):

                if "pnl" in infos:
                    pnl_roll.append(np.mean(infos["pnl"]))

                if "cost" in infos:
                    cost_roll.append(np.mean(infos["cost"]))

                if "slippage" in infos:
                    slip_roll.append(np.mean(infos["slippage"]))

                if "risk_pen" in infos:
                    risk_roll.append(np.mean(infos["risk_pen"]))

                if "equity" in infos:
                    equity_roll.append(np.mean(infos["equity"]))

                if "drawdown" in infos:
                    drawdown_roll.append(np.mean(infos["drawdown"]))

                if "position" in infos:
                    position_roll.append(np.mean(infos["position"]))

                if "turnover" in infos:
                    ep_turnover += infos["turnover"]

            # -------- episode tracking --------

            ep_returns += reward

            if done_env.any():

                finished = np.where(done_env)[0]

                ep_history.extend(ep_returns[finished].tolist())
                turnover_history.extend(ep_turnover[finished].tolist())

                ep_returns[finished] = 0
                ep_turnover[finished] = 0
                # IMPORTANT: reset only finished envs
                next_obs, _ = env.reset(options={"reset_mask": done_env})

            obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)

        # -------- bootstrap value --------

        with torch.no_grad():
            _, last_value = model(obs)

        returns, adv = compute_gae(
            rew_buf,
            done_buf,
            val_buf,
            last_value.squeeze(-1),
            gamma,
            lam
        )

        # -------- flatten rollout --------

        B = n_steps * num_envs

        obs_batch = obs_buf.reshape(B, obs_dim)
        u_batch = u_buf.reshape(B, act_dim)

        old_logp = logp_buf.reshape(B)
        ret_batch = returns.reshape(B)

        adv_batch = adv.reshape(B)

        # normalize advantages
        adv_batch = (adv_batch - adv_batch.mean()) / (adv_batch.std() + 1e-8)

        idx = torch.arange(B, device=device)

        stop = False
        last_kl = 0.0

        for _ in range(ppo_epochs):

            perm = idx[torch.randperm(B)]

            for start in range(0, B, minibatch_size):

                mb = perm[start:start + minibatch_size]

                dist, value = model(obs_batch[mb])

                logp = logprob_squashed(dist, u_batch[mb])

                entropy = dist.entropy().sum(-1)
                 # Approximate KL
                approx_kl = (old_logp[mb] - logp).mean().detach()
                last_kl = approx_kl.item()

                if last_kl > target_kl:
                    stop = True
                    break

                ratio = torch.exp(logp - old_logp[mb])

                # PPO clipped policy objective

                unclipped = ratio * adv_batch[mb]

                clipped = torch.clamp(
                    ratio,
                    1 - clip_eps,
                    1 + clip_eps
                ) * adv_batch[mb]

                policy_loss = -torch.min(unclipped, clipped).mean()

                # Value loss (simple version; can later replace with clipped value loss)
                value_loss = (ret_batch[mb] - value.squeeze(-1)).pow(2).mean()


                # Entropy bonus

                entropy_loss = -entropy.mean()

                loss = (
                    policy_loss
                    + vf_coef * value_loss
                    + ent_coef * entropy_loss
                )

                optimizer.zero_grad()
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_grad_norm
                )

                optimizer.step()

            if stop:
                break

        # keep std stable
        with torch.no_grad():
            model.log_std.clamp_(-1.5, -0.5)

        # -------- W&B logging --------

        mean_return = np.mean(ep_history[-100:]) if len(ep_history) >= 100 else np.nan
        mean_turnover = np.mean(turnover_history[-100:]) if len(turnover_history) >= 100 else np.nan

        wandb.log({

            "update": update,

            "mean_return_100": mean_return,
            "turnover_100": mean_turnover,

            "rollout_pnl": np.mean(pnl_roll) if pnl_roll else np.nan,
            "rollout_cost": np.mean(cost_roll) if cost_roll else np.nan,
            "rollout_slippage": np.mean(slip_roll) if slip_roll else np.nan,
            "rollout_risk_pen": np.mean(risk_roll) if risk_roll else np.nan,

            "equity": np.mean(equity_roll) if equity_roll else np.nan,
            "drawdown": np.mean(drawdown_roll) if drawdown_roll else np.nan,
            "position": np.mean(position_roll) if position_roll else np.nan,

            "kl": last_kl,
            "log_std": model.log_std.mean().item()

        })

    env.close()

    # -------- FINAL EVAL --------

    final_eval = eval_policy(model, df_test, config, device)

    equity_curve, pos_curve = run_equity_curve(
        model,
        df_test,
        config,
        device
    )

    wandb.log({

        "equity_curve_eval": wandb.plot.line_series(
            xs=list(range(len(equity_curve))),
            ys=[equity_curve.tolist()],
            keys=["equity"],
            title="Equity Curve Greedy Eval",
            xname="step"
        )

    })

    wandb.log({

        "position_curve_eval": wandb.plot.line_series(
            xs=list(range(len(pos_curve))),
            ys=[pos_curve.tolist()],
            keys=["position"],
            title="Position Curve Greedy Eval",
            xname="step"
        )

    })

    wandb.log({
        "EVAL_mean_episode_reward": final_eval
    })

    print("FINAL EVAL:", final_eval)

# TRAINING & WANDB LOGS

In [20]:
seeds = [10]

for seed in seeds:

    run_config = dict(config)
    run_config["seed"] = seed

    wandb.init(
        project="PPO_Bitcoin_Extended_Env_Final",
        group=run_config["group"],
        name=f"{run_config['group']}_seed_{seed}",
        config=run_config,
        reinit=True,
    )

    train_agent(df_train, df_test, run_config)

    wandb.finish()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


FINAL EVAL: 17.646800752649934


EVAL_mean_episode_reward,▁
drawdown,▄▆▇▇▂▅▇▇▇▃▁█▅▆██▅▄▇▄▇▇▂▇██▇▆▁▇█▄▅▅▄▁▅███
equity,▃▂▂▂▅▂▅▄▅▂▂▄▂▂▂▁▂▄▃▆▂▂▃█▃▂▅▂▆▅▆▂▄▂▄▆▆▂▂▇
kl,▂▄▅▂▂▂▄▁▁▂▂▄▃▄▁▂▂▇▂▃▇▁▃▂▁▁▂▃▁█▄▄▁▃▆▁▁▃▂▃
log_std,██▇▇▇▇▇▇▇▇█▇▇██▇▆▆▆▆▆▇▇▆▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
mean_return_100,▁▃▃▄▄▃▄▄▅▅▄▄▄▅▄▄▄▅▅▆▆▅▅▄▄▄▄▅▄▄▅▆▇███
position,█▃▂▂▄▁▃▂▂▁▃▄▂▃▁▄▄▆▆▄▃▄▂▄▄▅▄▄▄▄▆▄▅▂▄▅▁▂▃▃
rollout_cost,█▆▇▆▃▂▁▅▃▃▃▂▂▇▄▃▂█▄▂▃▅▄▃▅▃▄▃▂▆▆▄▃▅▆▄▂▂▄▆
rollout_pnl,▅▄▄▄▇▁▆▄▁▃▂▇▅▃▇█▆█▆▇▆▄▂▄▃▆▃▁▁▄▂▂▂▆▅▁▇▇▁▂
rollout_risk_pen,▇▆▇▃▅▆▆█▆▁▄▁▅█▆▆▄▇▆▅▆▆▄▇▆▄██▃▆▇▄▅▃▇█▁▇▆▆
+3,...
